# 01. Preparación de Datos EURUSD

Este notebook se encarga de la carga, limpieza, preprocesamiento y generación de características (Feature Engineering) para los datos de EURUSD.
El objetivo es generar un dataset final listo para ser consumido por los modelos de entrenamiento, separando la lógica pesada de procesamiento.

**Salidas:**
- `../data/processed/EURUSD/df_final_features.parquet`: Dataset con features y targets.
- `../models/EURUSD/forex_candle_clusterer_eurusd_5mins.joblib`: Modelo de clustering ajustado.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import joblib
from datetime import timezone

# Ruta absoluta al directorio src
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.append(src_path)

from utils.exploration_utils import (
    read_csv_files,
    preprocess_data,
    generate_target_variable,
    engineer_features
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Carga y Preprocesamiento Inicial

In [2]:
# Read Data from CSV files:
folder_path = '../data/EURUSD/5mins'
print(f"Attempting to load data from CSV files in: {folder_path}")

try:
    if not os.path.isdir(folder_path):
         raise FileNotFoundError(f"Error: Directory not found at {folder_path}. Please check the path.")

    df_EURUSD_5min = read_csv_files(folder_path)

    if df_EURUSD_5min.empty:
        print("Warning: No data loaded. The DataFrame is empty.")
    else:
        print(f"Data loaded successfully from CSVs in {folder_path}")
        print(f"Initial DataFrame shape: {df_EURUSD_5min.shape}")

        print("Applying preprocessing (preprocess_data)...")
        df_EURUSD_5min = preprocess_data(df_EURUSD_5min)
        print(f"DataFrame shape after preprocess_data: {df_EURUSD_5min.shape}")

except Exception as e:
    print(f"An error occurred: {e}")
    raise

Attempting to load data from CSV files in: ../data/EURUSD/5mins
Data loaded successfully from CSVs in ../data/EURUSD/5mins
Initial DataFrame shape: (1105411, 7)
Applying preprocessing (preprocess_data)...
Starting preprocessing...


c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:33: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['datetime_localized'] = pd.to_datetime(df['date'], format='%Y%m%d %H:%M:%S %Z', errors='coerce', utc=False)
c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:33: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Series constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  df['datetime_localized'] = pd.to_datetime(df['date'], format='%Y%m%d %H:%M:%S %Z', errors='coerce', utc=False)


Convirtiendo fechas/horas parseadas a UTC...
Extrayendo componentes de fecha/hora desde UTC...
Ordenando DataFrame por fecha/hora UTC...
Preprocesamiento completado. DataFrame ordenado por fecha/hora UTC.
DataFrame shape after preprocess_data: (1105411, 14)


## 2. Generación de Targets

In [3]:
TARGET_VAR_BUY = 'target_buy'
TARGET_VAR_SELL = 'target_sell'

# Ordenar por tiempo (seguridad)
df_EURUSD_5min = df_EURUSD_5min.sort_values('datetime').reset_index(drop=True)

PIPS_OBJETIVO_GENERACION = 25 
PIPS_STOP_GENERACION = 10    
FUTURE_PERIODS = 18          

# Generar targets (modifica df_processed)
df_processed = generate_target_variable(df_EURUSD_5min.copy(), 
                                        pips_objetivo=PIPS_OBJETIVO_GENERACION, 
                                        pips_stop=PIPS_STOP_GENERACION, 
                                        future_periods=FUTURE_PERIODS)

# Crear Targets Binarios
df_processed[TARGET_VAR_BUY] = (df_processed['compra_resultado'] == 'OBJETIVO_ALCANZADO').astype(int)
df_processed[TARGET_VAR_SELL] = (df_processed['venta_resultado'] == 'OBJETIVO_ALCANZADO').astype(int)

# Limpieza inicial
df_processed = df_processed.dropna(subset=['compra_resultado', 'venta_resultado'])
df_processed = df_processed.dropna(subset=['future_high_values', 'future_low_values', 'future_close_values'], how='any')

print("Targets generados.")

Generating target variable (pips_objetivo=25, pips_stop=10, future_periods=18)...
Evaluating trading scenarios...
Target variable 'multiclass_target' created.
Targets generados.


## 3. Split Train/Test/OOT
Aunque el Feature Engineering se aplica a todo (cuidando el leakage), definimos los rangos aquí para referencia o debug.

In [4]:
# Definir Fechas de Corte
split_date_train_end = "2023-12-31" 
split_date_test_end = "2024-12-31"

if df_processed['datetime'].dt.tz is not None:
    tz_info = df_processed['datetime'].dt.tz
    split_datetime_train_end = pd.Timestamp(split_date_train_end, tz=tz_info).replace(hour=23, minute=59, second=59)
    split_datetime_test_end = pd.Timestamp(split_date_test_end, tz=tz_info).replace(hour=23, minute=59, second=59)
else:
    split_datetime_train_end = pd.Timestamp(split_date_train_end).replace(hour=23, minute=59, second=59)
    split_datetime_test_end = pd.Timestamp(split_date_test_end).replace(hour=23, minute=59, second=59)

df_train = df_processed[df_processed['datetime'] <= split_datetime_train_end].copy()
df_test = df_processed[(df_processed['datetime'] > split_datetime_train_end) & (df_processed['datetime'] <= split_datetime_test_end)].copy()
df_oot = df_processed[df_processed['datetime'] > split_datetime_test_end].copy()

print(f"   Train rows: {len(df_train)}")
print(f"   Test rows:  {len(df_test)}")
print(f"   OOT rows:   {len(df_oot)}")

# --- Exploración de Targets y Fechas por Set ---
datasets = {
    "TRAIN": df_train,
    "TEST": df_test,
    "OOT": df_oot
}

print("\n--- Distribución y Rangos de Datos ---")
for name, df_set in datasets.items():
    print(f"\n{name}: {len(df_set)} filas")
    if len(df_set) > 0:
        # Fechas
        start_date = df_set['datetime'].min()
        end_date = df_set['datetime'].max()
        print(f"   Rango: {start_date}  -->  {end_date}")

        # Distribución de Targets
        buy_pct = df_set[TARGET_VAR_BUY].value_counts(normalize=True).get(1, 0) * 100
        sell_pct = df_set[TARGET_VAR_SELL].value_counts(normalize=True).get(1, 0) * 100
        
        print(f"   BUY  Positivo: {buy_pct:.2f}%")
        print(f"   SELL Positivo: {sell_pct:.2f}%")
    else:
        print("   (Dataset Vacío)")

   Train rows: 1009598
   Test rows:  72449
   OOT rows:   23364

--- Distribución y Rangos de Datos ---

TRAIN: 1009598 filas
   Rango: 2010-01-03 22:15:00+00:00  -->  2023-12-29 20:55:00+00:00
   BUY  Positivo: 7.11%
   SELL Positivo: 7.23%

TEST: 72449 filas
   Rango: 2024-01-02 00:00:00+00:00  -->  2024-12-31 21:55:00+00:00
   BUY  Positivo: 2.32%
   SELL Positivo: 2.69%

OOT: 23364 filas
   Rango: 2025-01-02 00:00:00+00:00  -->  2025-04-25 14:20:00+00:00
   BUY  Positivo: 8.43%
   SELL Positivo: 6.94%


## 4. Feature Engineering Robusto
Aplicamos ingeniería de características. Ajustamos el clusterer SOLO en Train y transformamos todo.

In [5]:
N_CANDLES_PARAM = 15
N_CLUSTERS_PARAM = 5
model_dir = '../models/EURUSD'
processed_data_dir = '../data/processed/EURUSD'
clusterer_filepath = os.path.join(model_dir, 'forex_candle_clusterer_eurusd_5mins.joblib')
final_features_path = os.path.join(processed_data_dir, 'df_final_features.parquet')

if not os.path.exists(model_dir): os.makedirs(model_dir)
if not os.path.exists(processed_data_dir): os.makedirs(processed_data_dir)

# 1. Fit en TRAIN
print("Starting Feature Engineering on TRAIN (Fitting Clusterer)...")
df_train_engineered, fitted_clusterer = engineer_features(
    df_train, n_candles=N_CANDLES_PARAM, n_clusters=N_CLUSTERS_PARAM, 
    min_pattern_threshold=100, fit_clusterer=True, random_state=42
)

# Guardar el clusterer inmediatamente
joblib.dump(fitted_clusterer, clusterer_filepath)
print(f"Clusterer saved to {clusterer_filepath}")

# 2. Transform en TEST y OOT (usando el clusterer ya fitteado)
print("Applying Feature Engineering on TEST (Transforming)...")
# Parámetro corregido: 'clusterer' en lugar de 'clusterer_model'

df_test_engineered, _ = engineer_features(
    df_test, n_candles=N_CANDLES_PARAM, n_clusters=N_CLUSTERS_PARAM, 
    fit_clusterer=False, clusterer=fitted_clusterer
)

print("Applying Feature Engineering on OOT (Transforming)...")
df_oot_engineered, _ = engineer_features(
    df_oot, n_candles=N_CANDLES_PARAM, n_clusters=N_CLUSTERS_PARAM, 
    fit_clusterer=False, clusterer=fitted_clusterer
)

# 3. Concatenar todo para guardar un solo archivo grande (o guardar separados, a gusto. Guardar junto facilita carga)
df_final = pd.concat([df_train_engineered, df_test_engineered, df_oot_engineered], axis=0)
df_final = df_final.sort_index() # O sort_values('datetime') si el indice no es tiempo

# 4. Limpieza Final de Columnas no deseadas antes de guardar (opcional, aunque mejor dejar limpieza fina al modelo)
# Pero eliminaremos las columnas muy pesadas o 'future_' que no sean targets finales si pesan mucho.
# Mantenemos targets.
cols_to_drop = [
    'future_high_values', 'future_low_values', 'future_close_values'
]
df_final = df_final.drop(columns=cols_to_drop, errors='ignore')

print(f"Final DataFrame shape: {df_final.shape}")

Starting Feature Engineering on TRAIN (Fitting Clusterer)...
--- Starting Feature Engineering ---
>>> Starting step: apply_technical_indicators
<<< Finished step: apply_technical_indicators
>>> Starting step: apply_candle_features (n_candles=15)


c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:126: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_range'].fillna(0, inplace=True)
c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:127: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the 

<<< Finished step: apply_candle_features
   Fitting and transforming with NEW CandleClusterer...
>>> Starting step: CandleClusterer.fit (k=5)
   KMeans fitted.
   Fitting pattern mapping...
   Pattern mapping created. 0 rare patterns will be mapped.
<<< Finished step: CandleClusterer.fit
>>> Starting step: CandleClusterer.transform
   Candle clusters predicted.
   Basic patterns created.
   No pattern mapping to apply (or mapping is empty).
<<< Finished step: CandleClusterer.transform
>>> Starting step: replace_inf_with_nan
  Columns initially identified as numeric: ['open', 'high', 'low', 'close', 'volume', 'barCount', 'Year', 'Day', 'Hour', 'Minute', 'DayOfWeek', 'compra_velas', 'compra_pips', 'venta_velas', 'venta_pips', 'target_buy', 'target_sell', 'SMA_10', 'SMA_40', 'SMA_100', 'diff_SMA_10_SMA_40', 'ratio_SMA_10_SMA_40', 'ratio_SMA_40_SMA_10', 'diff_SMA_10_SMA_100', 'ratio_SMA_10_SMA_100', 'ratio_SMA_100_SMA_10', 'diff_SMA_40_SMA_100', 'ratio_SMA_40_SMA_100', 'ratio_SMA_100_SMA_4

c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:126: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_range'].fillna(0, inplace=True)
c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:127: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the 

<<< Finished step: apply_candle_features
   Transforming with PROVIDED CandleClusterer...
>>> Starting step: CandleClusterer.transform
   Candle clusters predicted.
   Basic patterns created.
   No pattern mapping to apply (or mapping is empty).
<<< Finished step: CandleClusterer.transform
>>> Starting step: replace_inf_with_nan
  Columns initially identified as numeric: ['open', 'high', 'low', 'close', 'volume', 'barCount', 'Year', 'Day', 'Hour', 'Minute', 'DayOfWeek', 'compra_velas', 'compra_pips', 'venta_velas', 'venta_pips', 'target_buy', 'target_sell', 'SMA_10', 'SMA_40', 'SMA_100', 'diff_SMA_10_SMA_40', 'ratio_SMA_10_SMA_40', 'ratio_SMA_40_SMA_10', 'diff_SMA_10_SMA_100', 'ratio_SMA_10_SMA_100', 'ratio_SMA_100_SMA_10', 'diff_SMA_40_SMA_100', 'ratio_SMA_40_SMA_100', 'ratio_SMA_100_SMA_40', 'sma_order_score', 'slope_SMA_10', 'slope_SMA_40', 'slope_SMA_100', 'diff_slope_SMA_10_slope_SMA_40', 'diff_slope_SMA_10_slope_SMA_100', 'diff_slope_SMA_40_slope_SMA_100', 'norm_diff_SMA_10_SMA_1

c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:126: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_range'].fillna(0, inplace=True)
c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\src\utils\exploration_utils.py:127: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the 

   Basic patterns created.
   No pattern mapping to apply (or mapping is empty).
<<< Finished step: CandleClusterer.transform
>>> Starting step: replace_inf_with_nan
  Columns initially identified as numeric: ['open', 'high', 'low', 'close', 'volume', 'barCount', 'Year', 'Day', 'Hour', 'Minute', 'DayOfWeek', 'compra_velas', 'compra_pips', 'venta_velas', 'venta_pips', 'target_buy', 'target_sell', 'SMA_10', 'SMA_40', 'SMA_100', 'diff_SMA_10_SMA_40', 'ratio_SMA_10_SMA_40', 'ratio_SMA_40_SMA_10', 'diff_SMA_10_SMA_100', 'ratio_SMA_10_SMA_100', 'ratio_SMA_100_SMA_10', 'diff_SMA_40_SMA_100', 'ratio_SMA_40_SMA_100', 'ratio_SMA_100_SMA_40', 'sma_order_score', 'slope_SMA_10', 'slope_SMA_40', 'slope_SMA_100', 'diff_slope_SMA_10_slope_SMA_40', 'diff_slope_SMA_10_slope_SMA_100', 'diff_slope_SMA_40_slope_SMA_100', 'norm_diff_SMA_10_SMA_100', 'norm_diff_SMA_40_SMA_100', 'close_diff_SMA_10', 'close_diff_SMA_40', 'close_diff_SMA_100', 'crossover', 'sma_spread', 'acc_SMA_10', 'acc_SMA_40', 'acc_SMA_100'

## 5. Guardar Datos Procesados

In [ ]:
print(f"Saving processed data to {final_features_path}...")

# Sanitize types to avoid PyArrow errors (especially with Period types)
for col in df_final.columns:
    if pd.api.types.is_object_dtype(df_final[col]) or pd.api.types.is_period_dtype(df_final[col]):
        print(f"Converting column {col} to string (was {df_final[col].dtype})")
        df_final[col] = df_final[col].astype(str)

df_final.to_parquet(final_features_path, index=False)
print("Done.")

Saving processed data to ../data/processed/EURUSD\df_final_features.parquet...
Converting column Month to string (was object)
Converting column Date to string (was object)
Converting column compra_resultado to string (was object)
Converting column venta_resultado to string (was object)
Converting column multiclass_target to string (was object)
Converting column candle_cluster to string (was object)
Converting column pattern to string (was object)


C:\Users\Administrador\AppData\Local\Temp\ipykernel_23060\426796206.py:5: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_object_dtype(df_final[col]) or pd.api.types.is_period_dtype(df_final[col]):
C:\Users\Administrador\AppData\Local\Temp\ipykernel_23060\426796206.py:5: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_object_dtype(df_final[col]) or pd.api.types.is_period_dtype(df_final[col]):
C:\Users\Administrador\AppData\Local\Temp\ipykernel_23060\426796206.py:5: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_object_dtype(df_final[col]) or pd.api.types.is_period_dtype(df_final[col]):
C:\Users\Administrador\AppData\Local\Temp\ipykernel_23060\426796206.py

Done.


: 